In [ ]:
from src.data.data import *
from src.embedor import *
from src.plotting import *
import matplotlib
import seaborn as sns
import umap
import numpy as np
from sklearn.manifold import TSNE, Isomap, SpectralEmbedding
import phate

%load_ext autoreload

In [ ]:
def get_low_energy_graph(embedor_object: EmbedOR, edge_pctile=33):
    """
    Returns a graph with edges that are below the specified percentile of edge distances.
    """
    G_low_energy = embedor_object.G.copy()
    edge_dists = {
        (u, v): embedor_object.G[u][v]['energy'] for u, v in embedor_object.G.edges
    }
    edge_distances = np.array(list(edge_dists.values()))
    for idx, (u, v) in enumerate(embedor_object.G.edges):
        if edge_distances[idx] > np.percentile(edge_distances, edge_pctile):
            G_low_energy.remove_edge(u, v)
    return G_low_energy

def eval_low_energy_edges(embedor_object: EmbedOR, edge_pctile=33, clusters=None):
    # get number of cluster bridging edges in the entire graph
    G_full = embedor_object.G.copy()
    n_brige_full = 0
    for edge in G_full.edges:
        u, v = edge
        if clusters[u] != clusters[v]:
            n_brige_full += 1
    # check what percent of low energy edges bridge the clusters
    G_low_energy = get_low_energy_graph(embedor_object, edge_pctile=edge_pctile)
    n_bridge_low_energy = 0
    for edge in G_low_energy.edges:
        u, v = edge
        if clusters[u] != clusters[v]:
            n_bridge_low_energy += 1
    return n_bridge_low_energy, n_brige_full


exp_params = {
    'p': 3,
}

In [ ]:
import scanpy as sc


# PBMC 10k dataset
pbmc_data = sc.datasets.pbmc68k_reduced()
sc.tl.pca(pbmc_data, svd_solver='arpack')
sc.pl.pca(pbmc_data, color='CST3')
sc.pl.pca_variance_ratio(pbmc_data, log=True)


pbmc_data_original = pbmc_data.copy()
# use PCA embeddings with 40 pcs
pbmc_data_X_pca = pbmc_data.obsm['X_pca'][:, :20]
pbmc_labels = pbmc_data.obs['bulk_labels'] # string

pbmc_labels_int, label_dict = pd.factorize(pbmc_labels)
unique_pbmc_labels_str = np.unique(pbmc_labels.to_numpy())

In [ ]:
embedor = EmbedOR(exp_params = {'p': 1, 'n_neighbors': 25})
embedding = embedor.fit_transform(pbmc_data_X_pca)

n_bridge_low_energy, n_bridge_full = eval_low_energy_edges(embedor, edge_pctile=33, clusters=pbmc_labels_int)
print(f"Number of cluster bridging edges in low energy graph: {n_bridge_low_energy}")
print(f"Number of cluster bridging edges in full graph: {n_bridge_full}")

In [ ]:
plot_graph_2D(embedding, embedor.G, title=None, node_color=pbmc_labels_int[embedor.G.nodes], edge_width=0.01, node_size=0.7, edge_color='grey')
plt.axis('on')
ax = plt.gca()
ax.set_xlabel('EmbedOR1')
ax.xaxis.label.set_size(20)
ax.set_ylabel('EmbedOR2')
ax.yaxis.label.set_size(20)
ax.spines['top'].set_visible(True)
ax.spines['right'].set_visible(True)
ax.spines['left'].set_visible(True)
ax.spines['bottom'].set_visible(True)
ax.set_xticks([])
ax.set_yticks([])
plt.savefig('pbmc_full_graph.png', dpi=1200, bbox_inches='tight')

In [ ]:
low_energy_graph = get_low_energy_graph(embedor, edge_pctile=33)
plot_graph_2D(embedding, low_energy_graph, title=None, node_color=pbmc_labels_int[low_energy_graph.nodes], edge_width=0.09, node_size=0.3, edge_color='green')
plt.axis('on')
ax = plt.gca()
ax.set_xlabel('EmbedOR1')
ax.xaxis.label.set_size(20)
ax.set_ylabel('EmbedOR2')
ax.yaxis.label.set_size(20)
ax.spines['top'].set_visible(True)
ax.spines['right'].set_visible(True)
ax.spines['left'].set_visible(True)
ax.spines['bottom'].set_visible(True)
ax.set_xticks([])
ax.set_yticks([])
plt.savefig('pbmc_low_energy_graph.png', dpi=1200, bbox_inches='tight')


In [ ]:
print(f' number of samples: {len(pbmc_data_X_pca)}')